# Chapter 18 — Claims, Evidence, and Decisions

**Companion to Applied AI**

Question: Why should a decision depend on evidence state rather than confident prose?

By the end of this notebook you will have:

- represented claims, evidence, and decisions as separate records
- shown two identical sentences with different evidence status
- demonstrated decisions snapshotting their basis and reporting basis_changed

## What this notebook demonstrates
`said ≠ supported ≠ relied-on`: a tiny claim graph where confident language carries no weight and decisions pin their basis.

In [1]:
SEED = 42
import random
random.seed(SEED)
print("seed:", SEED)

seed: 42


## 1. Claims are cheap; evidence has levels

In [2]:
claims = {
    "C1": {"text": "canary deploy is healthy"},
    "C2": {"text": "canary deploy is healthy"},  # identical prose
}
evidence = {
    "E1": {"claim": "C1", "level": "measured", "detail": "error rate 0.2% over 10k requests"},
    "E2": {"claim": "C2", "level": "asserted", "detail": "operator said so in chat"},
}
for cid, c in claims.items():
    lv = [e["level"] for e in evidence.values() if e["claim"] == cid]
    print(cid, repr(c["text"]), "-> evidence:", lv)

C1 'canary deploy is healthy' -> evidence: ['measured']
C2 'canary deploy is healthy' -> evidence: ['asserted']


## 2. Decisions snapshot their basis

In [3]:
decisions = {}
def record_decision(did: str, claim_id: str, action: str, ttl_s: int = 60):
    basis = sorted(eid for eid, e in evidence.items() if e["claim"] == claim_id)
    decisions[did] = {"claim": claim_id, "action": action, "basis": basis, "ttl_s": ttl_s}
    return decisions[did]

print(record_decision("D1", "C1", "promote canary"))
print(record_decision("D2", "C2", "promote canary"))

{'claim': 'C1', 'action': 'promote canary', 'basis': ['E1'], 'ttl_s': 60}
{'claim': 'C2', 'action': 'promote canary', 'basis': ['E2'], 'ttl_s': 60}


## 3. A refuting source arrives: which decisions rest on the shaken claim?

In [4]:
evidence["E3"] = {"claim": "C1", "level": "measured", "detail": "error rate 8% in last 5 min (refutes E1)"}
def basis_changed(did: str) -> bool:
    d = decisions[did]
    now = sorted(eid for eid, e in evidence.items() if e["claim"] == d["claim"])
    return now != d["basis"]

print("D1 basis_changed:", basis_changed("D1"), "(must be re-decided)")
print("D2 basis_changed:", basis_changed("D2"), "(identical prose, different claim, unaffected)")
assert basis_changed("D1") and not basis_changed("D2")

D1 basis_changed: True (must be re-decided)
D2 basis_changed: False (identical prose, different claim, unaffected)


## Interpretation
- Supports: identical sentences can carry different weight; decisions must pin the evidence they rest on and report when it moves.
- Does NOT support: a full provenance standard; levels here are coarse by design.

## Try it yourself
1. Expire D1's TTL and require fresh measurement before re-deciding.
2. Add an `agreement refused` case: two asserted-level sources that still cannot promote.
3. Query all decisions resting on C1 with one function.